# 0050 vs 009816 research
Run All after editing USER CONFIG. Add FINLAB_API_TOKEN in Colab Secrets. Approve Drive mount.
Only real post-launch ETF history; no hypothetical reconstruction. Short annualized results need caution.


In [ ]:
# USER CONFIG — only change this block for normal use
SYMBOLS = ["0050", "009816"]
BENCHMARK = "0050"
DATE_MODE = "intersection"
CUSTOM_START_DATE = None
CUSTOM_END_DATE = None
STRICT = False
RETURN_TYPE = "total_return"
RISK_FREE_RATE = 0.0
EXPORT_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/00Quant_Research/compare_stock_return"
USE_CACHE = True
REFRESH_CACHE = False
DISPLAY_SYMBOLS = None
FEES = {}
CUSTOM_ASSETS = {}
HOLDINGS_FILES = {}
NON_TRADING_DATES = {"0050": {"2025-06-11": "market_suspension", "2025-06-12": "market_suspension", "2025-06-13": "market_suspension", "2025-06-16": "market_suspension", "2025-06-17": "market_suspension"}}


## Install → import → Drive → authentication → environment


In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO = Path("/content/compare_stock_return")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/hh4832/compare_stock_return.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","."],check=True)
from compare_stock_return.config import Config
from compare_stock_return.pipeline import run
from compare_stock_return.data_loader import authenticate
from compare_stock_return.reporting import environment
from google.colab import drive
drive.mount("/content/drive")
if not EXPORT_TO_DRIVE:
    raise ValueError("This Colab notebook requires EXPORT_TO_DRIVE=True; use CLI for local output")
if not Path(DRIVE_OUTPUT_ROOT).is_dir():
    raise FileNotFoundError(f"Create this exact Google Drive folder first: {DRIVE_OUTPUT_ROOT}")
print("Output root:", DRIVE_OUTPUT_ROOT)
authenticate()
print(environment())


## Shared engine and common-period headline results


In [ ]:
config = Config(symbols=SYMBOLS, benchmark=BENCHMARK, date_mode=DATE_MODE,
    custom_start_date=CUSTOM_START_DATE, custom_end_date=CUSTOM_END_DATE, strict=STRICT,
    return_type=RETURN_TYPE, risk_free_rate=RISK_FREE_RATE, drive_output_root=DRIVE_OUTPUT_ROOT,
    use_cache=USE_CACHE, refresh_cache=REFRESH_CACHE, display_symbols=DISPLAY_SYMBOLS,
    fees=FEES, custom_assets=CUSTOM_ASSETS, holdings_files=HOLDINGS_FILES,
    non_trading_dates=NON_TRADING_DATES)
result = run(config)
from IPython.display import display, HTML
import pandas as pd
display(result["intersection_metrics"])
print("Full-history metrics are supplementary and use different date ranges.")
charts = result["run_path"] / "charts"
for name in ["normalized_equity_curve","underwater","rolling_12m_return","rolling_36m_return","rolling_36m_volatility","rolling_sharpe","rolling_excess_return"]:
    display(HTML((charts / f"{name}.html").read_text()))


## Benchmark, pairwise, drawdowns, fees and optional holdings
Holdings require official dated snapshots. Fee metadata is not an extra deduction. No linkage-fund NAV is assumed.


In [ ]:
for name in ["benchmark_metrics", "pairwise_metrics", "drawdowns", "monthly_excess_returns", "fees", "holdings_concentration", "holdings_overlap"]:
    path = result["run_path"] / "results" / f"{name}.csv"
    if path.exists() and path.stat().st_size > 1:
        print(name)
        display(pd.read_csv(path))
    else:
        print(name, "skipped / unavailable")
print(result["metadata"]["holdings_status"])
